# 02 - Yogyakarta LSTM Autoencoder Training

This notebook trains one multivariate LSTM Autoencoder on DI Yogyakarta daily climate sequences.

The model reconstructs 30-day weather windows. High reconstruction error indicates an unusual weather sequence. Warning type is interpreted later by physical rainfall and wind rules.


In [ ]:
from pathlib import Path
import json
import random

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import RobustScaler
from tensorflow.keras import Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input, RepeatVector, TimeDistributed
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("..").resolve()
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed" / "yogyakarta_weather_features.csv"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
REPORT_DIR = PROJECT_ROOT / "reports"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

SEQUENCE_LENGTH = 30
BATCH_SIZE = 32
MAX_EPOCHS = 200
LEARNING_RATE = 0.001
EARLY_STOPPING_PATIENCE = 15
VALIDATION_FRACTION = 0.15
TEST_FRACTION = 0.15


In [ ]:
model_features = [
    "Tn",
    "Tx",
    "Tavg",
    "RH_avg",
    "RR",
    "ss",
    "ff_x",
    "ff_avg",
    "temp_range",
    "rain_3d",
    "rain_7d",
    "rain_change_1d",
    "wind_change_1d",
    "ddd_x_sin",
    "ddd_x_cos",
    "day_of_year_sin",
    "day_of_year_cos",
    "station_96855",
    "missing_RR",
    "missing_ff_x",
    "missing_ff_avg",
    "missing_RH_avg",
]

feature_weights = {
    "RR": 3.0,
    "rain_3d": 3.0,
    "rain_7d": 3.0,
    "ff_x": 2.5,
    "ff_avg": 2.5,
    "wind_change_1d": 2.5,
    "RH_avg": 1.5,
    "Tavg": 1.0,
    "Tn": 1.0,
    "Tx": 1.0,
    "temp_range": 1.0,
    "ss": 1.0,
    "ddd_x_sin": 0.8,
    "ddd_x_cos": 0.8,
    "day_of_year_sin": 0.5,
    "day_of_year_cos": 0.5,
    "station_96855": 0.5,
    "missing_RR": 0.5,
    "missing_ff_x": 0.5,
    "missing_ff_avg": 0.5,
    "missing_RH_avg": 0.5,
}


In [ ]:
data = pd.read_csv(PROCESSED_PATH, parse_dates=["date"], dtype={"station_id": "string"})
data = data.sort_values(["station_id", "date"]).reset_index(drop=True)

print("Processed shape:", data.shape)
display(data.head())
display(data.groupby("station_id")["date"].agg(["min", "max", "count"]))


In [ ]:
ordered_dates = np.array(sorted(data["date"].unique()))
n_dates = len(ordered_dates)
test_size = int(round(n_dates * TEST_FRACTION))
validation_size = int(round(n_dates * VALIDATION_FRACTION))
train_end = n_dates - validation_size - test_size
validation_end = n_dates - test_size

train_dates = ordered_dates[:train_end]
validation_dates = ordered_dates[train_end:validation_end]
test_dates = ordered_dates[validation_end:]

train_df = data[data["date"].isin(train_dates)].copy()
validation_df = data[data["date"].isin(validation_dates)].copy()
test_df = data[data["date"].isin(test_dates)].copy()

print("Train:", train_df["date"].min().date(), "to", train_df["date"].max().date(), train_df.shape)
print("Validation:", validation_df["date"].min().date(), "to", validation_df["date"].max().date(), validation_df.shape)
print("Test:", test_df["date"].min().date(), "to", test_df["date"].max().date(), test_df.shape)


In [ ]:
scaler = RobustScaler()
scaler.fit(train_df[model_features])

def scale_frame(df: pd.DataFrame) -> pd.DataFrame:
    scaled = df.copy()
    scaled[model_features] = scaler.transform(scaled[model_features])
    return scaled

train_scaled = scale_frame(train_df)
validation_scaled = scale_frame(validation_df)
test_scaled = scale_frame(test_df)


In [ ]:
def build_sequences(df: pd.DataFrame, feature_names: list, sequence_length: int):
    x_values = []
    metadata_rows = []
    skipped_windows = 0

    for station_id, group in df.sort_values(["station_id", "date"]).groupby("station_id", sort=False):
        g = group.sort_values("date").reset_index(drop=True)
        values = g[feature_names].to_numpy(dtype=np.float32)

        for end_idx in range(sequence_length - 1, len(g)):
            start_idx = end_idx - sequence_length + 1
            window_dates = g.loc[start_idx:end_idx, "date"]
            day_differences = window_dates.diff().dt.days.iloc[1:]

            if not (day_differences == 1).all():
                skipped_windows += 1
                continue

            x_values.append(values[start_idx:end_idx + 1])
            metadata_rows.append(
                {
                    "date": g.loc[end_idx, "date"],
                    "station_id": str(station_id),
                    "station_name": g.loc[end_idx, "station_name"],
                    "region_name": g.loc[end_idx, "region_name"],
                }
            )

    print("Skipped non-consecutive date windows:", skipped_windows)

    if not x_values:
        return np.empty((0, sequence_length, len(feature_names)), dtype=np.float32), pd.DataFrame(metadata_rows)

    return np.stack(x_values).astype(np.float32), pd.DataFrame(metadata_rows)


x_train, train_meta = build_sequences(train_scaled, model_features, SEQUENCE_LENGTH)
x_validation, validation_meta = build_sequences(validation_scaled, model_features, SEQUENCE_LENGTH)
x_test, test_meta = build_sequences(test_scaled, model_features, SEQUENCE_LENGTH)

print("x_train:", x_train.shape)
print("x_validation:", x_validation.shape)
print("x_test:", x_test.shape)
display(train_meta.head())


In [ ]:
def build_lstm_autoencoder(sequence_length: int, n_features: int, learning_rate: float) -> Model:
    inputs = Input(shape=(sequence_length, n_features))
    encoded = LSTM(64, activation="tanh", return_sequences=True)(inputs)
    encoded = Dropout(0.2)(encoded)
    encoded = LSTM(32, activation="tanh", return_sequences=False)(encoded)

    decoded = RepeatVector(sequence_length)(encoded)
    decoded = LSTM(32, activation="tanh", return_sequences=True)(decoded)
    decoded = Dropout(0.2)(decoded)
    decoded = LSTM(64, activation="tanh", return_sequences=True)(decoded)
    outputs = TimeDistributed(Dense(n_features))(decoded)

    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss="mse")
    return model


model = build_lstm_autoencoder(SEQUENCE_LENGTH, len(model_features), LEARNING_RATE)
model.summary()


In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=EARLY_STOPPING_PATIENCE,
    restore_best_weights=True,
)

history = model.fit(
    x_train,
    x_train,
    validation_data=(x_validation, x_validation),
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stopping],
    shuffle=False,
    verbose=1,
)


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history.history["loss"], label="train_loss")
plt.plot(history.history["val_loss"], label="val_loss")
plt.title("LSTM Autoencoder Training Curve")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
def weight_vector(feature_names: list) -> np.ndarray:
    weights = np.array([feature_weights.get(name, 1.0) for name in feature_names], dtype=np.float32)
    return weights / weights.mean()


def weighted_reconstruction_error(x_true: np.ndarray, x_pred: np.ndarray, feature_names: list) -> np.ndarray:
    weights = weight_vector(feature_names).reshape(1, 1, -1)
    squared_error = np.square(x_true - x_pred)
    return np.mean(squared_error * weights, axis=(1, 2))


validation_pred = model.predict(x_validation, verbose=0)
validation_scores = weighted_reconstruction_error(x_validation, validation_pred, model_features)

thresholds = {
    "p95": float(np.percentile(validation_scores, 95)),
    "p99": float(np.percentile(validation_scores, 99)),
    "p995": float(np.percentile(validation_scores, 99.5)),
}

print(thresholds)


In [ ]:
model.save(ARTIFACT_DIR / "model_lstm_autoencoder.keras")
joblib.dump(scaler, ARTIFACT_DIR / "scaler.pkl")

feature_config = {
    "sequence_length": SEQUENCE_LENGTH,
    "model_features": model_features,
    "feature_weights": feature_weights,
    "training_stations": ["96851", "96855"],
}

training_report = {
    "train_sequences": int(len(x_train)),
    "validation_sequences": int(len(x_validation)),
    "test_sequences": int(len(x_test)),
    "thresholds": thresholds,
    "loss_history": {key: [float(v) for v in values] for key, values in history.history.items()},
}

(ARTIFACT_DIR / "threshold.json").write_text(json.dumps(thresholds, indent=2), encoding="utf-8")
(ARTIFACT_DIR / "feature_config.json").write_text(json.dumps(feature_config, indent=2), encoding="utf-8")
(ARTIFACT_DIR / "training_report.json").write_text(json.dumps(training_report, indent=2), encoding="utf-8")

print("Saved model and artifacts to:", ARTIFACT_DIR)
